# ReAct Agents: Reasoning and Acting with Language Models

## Introduction to ReAct

**ReAct** (Reasoning + Acting) is a powerful paradigm that combines reasoning and acting in language models. Unlike traditional chatbots that only generate text, ReAct agents can:

- 🧠 **Reason** through problems step by step
- 🔧 **Act** by using external tools and APIs
- 🔍 **Observe** the results of their actions
- 🔄 **Iterate** until they reach a solution

### What You'll Learn

In this notebook, we'll explore:
1. **Memory Management** - Cleaning up GPU memory for optimal performance
2. **OpenAI Integration** - Setting up cloud-based LLM for agent reasoning
3. **ReAct Prompting** - Understanding the reasoning-action-observation cycle
4. **Tool Integration** - Adding web search and mathematical calculation capabilities
5. **Agent Execution** - Running complex multi-step queries

### Why ReAct Matters

ReAct agents represent a significant advancement in AI capabilities:
- **Problem Solving**: Can break down complex queries into manageable steps
- **Tool Usage**: Leverage external APIs and services
- **Transparency**: Show their reasoning process
- **Reliability**: Can verify and correct their own work

---


## Step 1: Memory Management

### Why Clean Memory?

Before working with agents, it's crucial to clean GPU memory, especially on Apple Silicon devices. ReAct agents can be memory-intensive because they:

- 🔄 **Make multiple LLM calls** during reasoning cycles
- 📊 **Process complex prompts** with tool descriptions and conversation history
- 🧠 **Maintain state** across multiple reasoning steps

### The Cleanup Function

This utility function ensures optimal performance by:
- **Clearing GPU cache** - Prevents memory fragmentation
- **Removing global variables** - Cleans up any previously loaded models
- **Forcing garbage collection** - Frees up system memory

**Best Practice**: Always run this before starting agent workflows.


In [2]:
def cleanup_mps_memory():
    """
    Frees MPS memory by deleting global variables 'model' and 'tokenizer' if they exist.
    Useful when you want to avoid passing model/tokenizer manually.
    """
    import gc
    import torch

    for var in ['model', 'tokenizer', 'pipe']:
        if var in globals():
            print(f"🔹 Deleting: {var}")
            del globals()[var]

    gc.collect()
    torch.mps.empty_cache()
    print("MPS memory cleaned.")
cleanup_mps_memory()

MPS memory cleaned.


## Step 2: Setting Up the Language Model

### Why Use OpenAI for Agents?

While local models can work for simple tasks, **cloud-based models like GPT-3.5-turbo** offer several advantages for ReAct agents:

- 🎯 **Better Reasoning**: Superior logical reasoning and planning capabilities
- 🔧 **Tool Understanding**: Better comprehension of when and how to use tools
- 📝 **Format Adherence**: More reliable at following the ReAct prompt format
- ⚡ **Speed**: Faster inference for real-time agent interactions

### Configuration Details

- **Model**: `gpt-3.5-turbo` - Good balance of performance and cost
- **Temperature**: `0` - Ensures deterministic, focused responses (crucial for agents)
- **API Key**: Loaded securely from environment variables

### Environment Setup

Make sure your `.env` file contains:
```
OPENAI_API_KEY=your_api_key_here
```

**Security Note**: Never hardcode API keys in your notebooks!


In [3]:
import openai
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# Create client
client = openai.OpenAI(api_key=api_key)

In [8]:
from langchain_openai import ChatOpenAI
openai_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

## Step 3: Understanding the ReAct Prompt Template

### The ReAct Framework

The ReAct prompt template is the **heart of the agent system**. It defines a structured thinking process that alternates between reasoning and acting:

```
Question → Thought → Action → Observation → Thought → Action → ... → Final Answer
```

### Breaking Down the Template

#### 1. **Tool Description Section**
```
You have access to the following tools: {tools}
```
- Lists all available tools and their descriptions
- Helps the agent understand what capabilities it has

#### 2. **Format Instructions**
The template enforces a specific format:
- **Question**: The original user query
- **Thought**: Agent's reasoning about what to do next
- **Action**: Which tool to use
- **Action Input**: Parameters to pass to the tool
- **Observation**: Results from the tool execution
- **Final Answer**: The conclusive response

#### 3. **Key Variables**
- `{tools}`: Descriptions of available tools
- `{tool_names}`: List of tool names for action selection
- `{input}`: The user's question
- `{agent_scratchpad}`: Conversation history and previous reasoning

### Why This Structure Works

- 🔍 **Transparency**: Shows the agent's thinking process
- 🎯 **Guided Reasoning**: Forces systematic problem-solving
- 🔧 **Tool Selection**: Clear framework for choosing appropriate tools
- 🔄 **Iterative Process**: Allows multiple reasoning-action cycles


In [4]:
from langchain import PromptTemplate

## Step 4: Setting Up Agent Tools

### What Are Agent Tools?

**Tools** are external capabilities that extend the agent's abilities beyond text generation. They allow agents to:

- 🔍 **Search the web** for current information
- 🧮 **Perform calculations** with mathematical precision
- 📊 **Query databases** for specific data
- 🌐 **Call APIs** for specialized services

### Tool Configuration

#### 1. **Web Search Tool** - DuckDuckGo
```python
search_tool = Tool(
    name="duckduck",
    description="A web search engine. Use this as a search engine for general queries.",
    func=search.run,
)
```

**Why DuckDuckGo?**
- **Privacy-focused**: No tracking or personal data collection
- **Free**: No API costs or rate limits
- **Reliable**: Good search results for general queries
- **Easy Integration**: Simple setup with LangChain

#### 2. **Math Tool** - LLM-powered Calculator
```python
tools = load_tools(["llm-math"], llm=openai_llm)
```

**Features:**
- **Complex Calculations**: Handles advanced mathematical operations
- **LLM Integration**: Uses the language model for mathematical reasoning
- **Python Execution**: Actually runs Python code for accuracy

### Tool Descriptions Matter

The `description` field is crucial because:
- 📝 **Agent Decision Making**: Helps the agent choose the right tool
- 🎯 **Usage Guidance**: Explains when and how to use each tool
- 🔧 **Parameter Hints**: Suggests what inputs the tool expects

**Best Practice**: Write clear, specific tool descriptions for better agent performance.


### Step 3.1: The ReAct Prompt in Detail

Let's examine what makes this prompt template so effective:

#### **Template Structure**
- **Tool Listing**: `{tools}` - Dynamic list of available capabilities
- **Format Specification**: Clear instructions for agent behavior
- **Example Flow**: Shows the expected reasoning pattern
- **Variables**: Placeholders for dynamic content

#### **Critical Elements**
1. **Action Constraints**: `should be one of [{tool_names}]` - Prevents invalid tool calls
2. **Iterative Structure**: Allows multiple reasoning cycles
3. **Clear Termination**: "I now know the final answer" signals completion
4. **Scratchpad**: `{agent_scratchpad}` maintains conversation state

This template is carefully designed to guide the agent through systematic problem-solving while maintaining flexibility for different types of queries.


In [5]:
# Create the ReAct template
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate(
    template=react_template,
    input_variables=["tools", "tool_names", "input", "agent_scratchpad"]
)

## Step 5: Creating and Running the ReAct Agent

### Agent Architecture

The ReAct agent consists of several key components:

#### 1. **Agent Creation**
```python
agent = create_react_agent(openai_llm, tools, prompt)
```
- **LLM**: The reasoning engine (GPT-3.5-turbo)
- **Tools**: Available capabilities (search + math)
- **Prompt**: The ReAct template defining behavior

#### 2. **Agent Executor**
```python
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)
```

**Key Parameters:**
- **`verbose=True`**: Shows the agent's step-by-step reasoning process
- **`handle_parsing_errors=True`**: Gracefully handles formatting mistakes
- **Tools Integration**: Connects the agent to its capabilities

### Understanding Agent Execution

When you run the agent with a query like "What is the current price of MacBook Pro M4?", here's what happens:

1. **Question Analysis**: Agent understands it needs current pricing information
2. **Tool Selection**: Realizes it needs to search the web (not in its training data)
3. **Action**: Uses the `duckduck` search tool
4. **Observation**: Processes the search results
5. **Reasoning**: Analyzes the information found
6. **Answer**: Provides a comprehensive response based on current data

### The Power of ReAct

Notice how the agent:
- 🧠 **Reasons** about what information it needs
- 🔍 **Acts** by searching for current prices
- 📊 **Observes** multiple search results
- 🎯 **Synthesizes** a helpful answer from real-time data

This demonstrates the agent's ability to handle queries that require **current information** beyond what was in its training data.


In [11]:
from langchain.agents import load_tools, Tool
from langchain.tools import DuckDuckGoSearchResults

search = DuckDuckGoSearchResults()
search_tool = Tool(
    name="duckduck",
    description="A web search engine. Use this as a search engine for general queries.",
    func=search.run,
)

tools = load_tools(["llm-math"], llm=openai_llm)
tools.append(search_tool)

### Analyzing the Agent's Reasoning Process

Let's break down what we can see in the agent's output:

#### 1. **Initial Reasoning** (Green Text)
```
I should use duckduck to search for the current price of MacBook Pro M4.
Action: duckduck
Action Input: "current price of MacBook Pro M4"
```

**What This Shows:**
- 🎯 **Problem Recognition**: Agent identifies it needs current information
- 🔧 **Tool Selection**: Chooses the appropriate search tool
- 📝 **Input Formulation**: Creates an effective search query

#### 2. **Search Results** (Yellow Text)
The agent receives multiple search results with:
- **Pricing Information**: Various retailer prices and discounts
- **Product Details**: Different M4 MacBook Pro configurations
- **Current Deals**: Special offers and price comparisons

#### 3. **Information Processing**
The agent must now:
- 📊 **Parse Multiple Sources**: Understand different price points
- 🔍 **Extract Key Information**: Identify the most relevant pricing
- 🎯 **Synthesize Response**: Provide a helpful summary

### Key Advantages of ReAct Agents

1. **Real-time Information**: Can access current data not in training
2. **Transparent Process**: Shows reasoning steps for verification
3. **Tool Flexibility**: Can combine multiple tools as needed
4. **Error Recovery**: Can retry or adjust approach if needed

### Common Use Cases

ReAct agents excel at:
- 📈 **Market Research**: Current prices, stock info, trends
- 🧮 **Complex Calculations**: Multi-step mathematical problems
- 🔍 **Fact Checking**: Verifying information against current sources
- 📊 **Data Analysis**: Combining search with calculations


In [12]:
from langchain.agents import AgentExecutor, create_react_agent
agent = create_react_agent(openai_llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

agent_executor.invoke({"input": "What is the current price of MacBook Pro M4?"})



> Entering new AgentExecutor chain...
I should use duckduck to search for the current price of MacBook Pro M4.
Action: duckduck
Action Input: "current price of MacBook Pro M4"snippet: The newly launched M4 Pro and M4 Max 14-inch MacBook Pros have shown notable performance improvements over their M1, M2, and M3 counterparts, especially in single-core scores. In recent benchmarks, the M4 Pro 14-inch MacBook Pro achieved a single-core score of approximately 3,850, surpassing the M3 Pro's single-core score by about 15-20%., title: Apple 14″ MacBook Pro Prices at MacPrices.net, link: https://www.macprices.net/14-macbook-pro/, snippet: Best 2024 14-inch MacBook Pro M4 Pro and M4 Max prices, with exclusive coupon discounts and easy price comparison. ... MacBook Pro 14-inch (M4 Pro): Advanced performance & exclusive discounts. The MacBook Pro 14-inch with the M4 Pro chip sets a new standard for power and efficiency. Featuring a 12-core CPU and 16-core GPU, it's designed to ..., title: MacBoo

{'input': 'What is the current price of MacBook Pro M4?',
 'output': 'The average price of the MacBook Pro M4 is $1551.67'}

## Summary: Building Intelligent ReAct Agents

### What We've Accomplished

This notebook demonstrated how to build a sophisticated ReAct agent that can:

#### 1. **Reason Through Problems** 🧠
- **Systematic Thinking**: Follows a structured thought process
- **Problem Decomposition**: Breaks complex queries into manageable steps
- **Decision Making**: Chooses appropriate tools based on the task

#### 2. **Act with Purpose** 🔧
- **Tool Selection**: Intelligently chooses between search and math tools
- **Parameter Formulation**: Creates effective inputs for each tool
- **Execution**: Actually performs external actions

#### 3. **Learn from Observations** 🔍
- **Result Processing**: Analyzes tool outputs
- **Information Synthesis**: Combines multiple sources
- **Iterative Improvement**: Can refine approach based on results

### Key Components We Covered

| Component | Purpose | Key Benefit |
|-----------|---------|-------------|
| **Memory Management** | Clean GPU resources | Optimal performance |
| **OpenAI Integration** | Powerful reasoning engine | Superior agent capabilities |
| **ReAct Template** | Structured thinking framework | Transparent reasoning |
| **Tool Configuration** | External capabilities | Extended functionality |
| **Agent Executor** | Orchestrates the process | Robust execution |

### ReAct vs Traditional Approaches

#### Traditional LLMs:
- ❌ **Static Knowledge**: Limited to training data
- ❌ **No Actions**: Can only generate text
- ❌ **Black Box**: Hidden reasoning process

#### ReAct Agents:
- ✅ **Dynamic Information**: Access to current data
- ✅ **Tool Usage**: Can perform actions
- ✅ **Transparent**: Shows step-by-step reasoning
- ✅ **Verifiable**: Can check and validate results

### Best Practices for ReAct Agents

1. **Tool Selection**
   - Choose tools that complement each other
   - Write clear, specific tool descriptions
   - Test tools independently before integration

2. **Prompt Engineering**
   - Use clear format instructions
   - Provide good examples
   - Handle edge cases gracefully

3. **Error Handling**
   - Enable parsing error recovery
   - Monitor agent performance
   - Implement fallback strategies

4. **Performance Optimization**
   - Clean memory before complex operations
   - Use appropriate model temperature (0 for deterministic behavior)
   - Monitor API usage and costs

### Advanced Extensions

To enhance your ReAct agents, consider adding:

- 🗄️ **Database Tools**: Query structured data
- 🌐 **API Integrations**: Weather, stock prices, news
- 📊 **Data Analysis**: pandas, numpy integration
- 🔒 **Authentication**: Secure API access
- 💾 **Memory Persistence**: Long-term conversation memory

### Real-World Applications

ReAct agents are perfect for:

- **Customer Support**: Research + knowledge base queries
- **Financial Analysis**: Market data + calculations
- **Research Assistance**: Literature search + summarization
- **E-commerce**: Product search + price comparisons
- **Content Creation**: Research + fact-checking

### Next Steps

1. **Experiment** with different tool combinations
2. **Optimize** prompts for your specific use case
3. **Monitor** performance and costs in production
4. **Extend** with custom tools for your domain
5. **Scale** to handle multiple concurrent users

**Congratulations! You now have the foundation to build intelligent, transparent, and capable AI agents! 🚀**
